<a href="https://colab.research.google.com/github/appling2024/MSP/blob/Liza_L/rules_task_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 23.4 MB/s eta 0:00:00


In [143]:
import nltk
nltk.download('punkt_tab')
from nltk.parse import FeatureEarleyChartParser
from nltk.grammar import FeatureGrammar
from nltk import *
import pymorphy3 as pm
import codecs

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


**Исходные правила:**

NUMRNP[C=?c] -> NUMR[C=?c] NP[C=?c, NUM=plur]

NUMRNP[C=accs] -> NUMR[C=accs] NP[C=gent]

NUMRNP[+nomn, C=nomn] -> NUMR[C=nomn] NP[C=gent]

NUMR[C=?c] -> NUMR[C=?c] NUMR[C=?c]

PP -> PREP NUMRNP[-nomn]

NP[+pp, C=?c, G=?g, NUM=?n] -> NP[C=?c, G=?g, NUM=?n] PP

VP[+datv, TENSE=?t, G=?g, NUM=?n, PER=?p] ->VP[TENSE=?t, G=?g, NUM=?n, PER=?p] NP[C=datv]

**Добавленные правила:**

NP[+numr, C=gent, G=?g, NUM=plur] -> NOUN[C=gent, G=?g, NUM=plur] NUMR[C=?c]

NUMRNP[C=?c] -> NUMR[C=?c]

VP[+gent, TENSE=?t, G=?g, NUM=?n, PER=?p] ->VP[TENSE=?t, G=?g, NUM=?n, PER=?p] NP[C=gent]

In [195]:
## загружаем PyMorphy2
m = pm.MorphAnalyzer()
## открываем (создаем)файл с грамматикой, куда будут записываться правила
f = codecs.open("test.fcfg", mode= "w", encoding = "utf-8")
rules = codecs.open("rules.txt", mode= "r", encoding = "utf-8")
## записываем правила, которые вручную делаем (некоторые на основе правил из АОТ)
for rule in rules:
    f.writelines(rules)
rules.close()
f.close()

In [117]:
## функция, которая переводит нужную нам информацию из пайморфи в вид, читаемый парсером NLTK
## принимает (токенизированное) словосочетание на входе, записывает правила (lexical productions) в тот же файл с грамматикой

In [196]:
def pm2fcfg (phrase): ## phrase - это словосочетание, которое мы разбираем
    f = codecs.open("test.fcfg", mode= "a", encoding = "utf-8")
    for x in phrase:
        a = m.parse(x) ## a - список возможных вариантов морфологического разбора слова, предлагаемых пайморфи
        ## от части речи зависит, какие признаки отправляются в грамматику, осюда условия
        for y in a:
            #print(y)
            if (y.tag.POS == "NOUN") or (y.tag.POS == "ADJF") or (y.tag.POS == "PRTF"):
                strk = str(y.tag.POS) + "[C=" + str(y.tag.case) + ", G=" + str(y.tag.gender) + ", NUM=" + str(y.tag.number) + ", PER=3" + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "ADJS") or (y.tag.POS == "PRTS"):
                strk = str(y.tag.POS) + "[G=" + str(y.tag.gender) + ", NUM=" + str(y.tag.number) + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "NUMR"):
                strk = str(y.tag.POS) + "[C=" + str(y.tag.case) + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "ADVB") or (y.tag.POS == "GRND") or (y.tag.POS == "COMP") or (y.tag.POS == "PRED") or (y.tag.POS == "PRCL") or (y.tag.POS == "INTJ"):
                strk = str(y.tag.POS) + "[NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "PREP") or (y.tag.POS == "CONJ"):
                strk = str(y.tag.POS) + "[NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
                break
            elif (y.tag.POS == "NPRO") & (y.normal_form != "это")& (y.normal_form != "нечего"):
                if ((y.tag.person[0] == "3") & (y.tag.number == "sing")):
                    strk = str(y.tag.POS) + "[C=" + str(y.tag.case) + ", G=" + str(y.tag.gender) + ", NUM=" + str(y.tag.number) + ", PER=" + str(y.tag.person)[0] + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                else:
                    strk = str(y.tag.POS) + "[C=" + str(y.tag.case) + ", NUM=" + str(y.tag.number) + ", PER=" + str(y.tag.person)[0] + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "VERB")  or (y.tag.POS == "INFN"):
                if (y.tag.tense == "past"):
                    strk = str(y.tag.POS) + "[TR=" + str(y.tag.transitivity) + ", TENSE=" + str(y.tag.tense) + ", G=" + str(y.tag.gender) + ", NUM=" + str(y.tag.number) + ", PER=" + "0" + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                elif (y.tag.POS == "INFN"):
                    strk = str(y.tag.POS) + "[TR=" + str(y.tag.transitivity) + ", TENSE=0, G=0, NUM=0, PER=0, NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                else:
                    strk = str(y.tag.POS) + "[TR=" + str(y.tag.transitivity) + ", TENSE=" + str(y.tag.tense) + ", G=" + "0" + ", NUM=" + str(y.tag.number) + ", PER=" + str(y.tag.person)[0] + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
    f.close()

In [192]:
text = input("введите предложение для разбора: ") ## сюда пишется словосочетание для разбора
words = nltk.word_tokenize(text) ## разбиваем словосочетание на токены

введите предложение для разбора: часа через два


In [197]:
pm2fcfg(words) ## запускаем функцию, описанную выше

with open("test.fcfg", "r", encoding="utf-8") as f:
    grammar = FeatureGrammar.fromstring(f.read())
parser = FeatureEarleyChartParser(grammar)
for tree in parser.parse(words):
    print(tree)

(XP[]
  (NP[C='gent', G='masc', NUM='sing', +pp]
    (NP[C='gent', G='masc', NUM='sing', PER=3]
      (NOUN[C='gent', G='masc', NF='час', NUM='sing', PER=3] часа))
    (PP[]
      (PREP[NF='через'] через)
      (NUMRNP[C='nomn'] (NUMR[C='nomn', NF='два'] два)))))
(XP[]
  (NP[C='gent', G='masc', NUM='sing', +pp]
    (NP[C='gent', G='masc', NUM='sing', PER=3]
      (NOUN[C='gent', G='masc', NF='час', NUM='sing', PER=3] часа))
    (PP[]
      (PREP[NF='через'] через)
      (NUMRNP[C='accs'] (NUMR[C='accs', NF='два'] два)))))
